# RQ3: what correlates with validation improvement and generalisation

**Definitions** (the same in every notebook, see the docstring of `analysis.py`):

- *improvement*: % change in CPU cycles of the returned patch on the validation inputs (negative = faster).
  A patch that fails the correctness oracle on the validation inputs counts as 0% (the original program is kept).
- *gap*: for each run, validation change minus training change, in percentage points (positive = the patch
  generalises worse than it trained). Aggregated with the median of the runs.
- *cell*: (software, algorithm, budget), 15 runs = 5 folds × 3 seeds. Each run has 1000 evaluations; the budgets
  100, 250, 500 and 750 are checkpoints of the same run.
- *ranking*: per software, the median of each cell, averaged over the five budgets and ranked; rank sum over the
  four systems (lower = better).

Spearman correlations of the budget, $V_t$ (share of evaluated variants that pass the tests) and $V_c$ (share that compile) with improvement and gap, per software (all algorithms pooled, `group = all`) and per algorithm. Every run and budget is used. With 120 correlations, $p$-values are adjusted with Holm and with Benjamini-Hochberg; the analysis is exploratory.

In [1]:
import pathlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(pathlib.Path('..', 'revision_analysis').resolve()))
import analysis as A

RESULTS = 'results.csv'           # one row per run and budget (checkpoint)
OUT = pathlib.Path('analysis')    # tables written by analysis.py
REFRESH = False                   # True: recompute every table from results.csv
if REFRESH or not (OUT / 'cells.csv').exists():
    A.run(RESULTS, OUT, 'zero')
df = A.load(RESULTS, 'zero')
cells = pd.read_csv(OUT / 'cells.csv')
SW = {'minisat_hack': 'MiniSAT-hack', 'sat4j': 'Sat4J', 'weka': 'Weka', 'pngOptim': 'OptiPNG'}
pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)

In [2]:
corr = pd.read_csv(OUT / 'rq3_correlations.csv')
display(corr[corr.group == 'all'])

,software,group,x,y,n,rho,p,p_holm,p_bh
0,minisat_hack,all,vt,improvement,300,1.05e-02,8.56e-01,1.00e+00,9.24e-01
1,minisat_hack,all,vt,gap,300,-7.76e-02,1.80e-01,1.00e+00,4.07e-01
2,minisat_hack,all,vc,improvement,300,9.69e-03,8.67e-01,1.00e+00,9.24e-01
3,minisat_hack,all,vc,gap,300,-7.74e-02,1.81e-01,1.00e+00,4.07e-01
4,minisat_hack,all,checkpoint,improvement,300,-5.28e-02,3.62e-01,1.00e+00,5.69e-01
5,minisat_hack,all,checkpoint,gap,300,1.30e-01,2.47e-02,1.00e+00,2.28e-01
6,pngOptim,all,vt,improvement,300,-1.11e-01,5.58e-02,1.00e+00,2.73e-01
7,pngOptim,all,vt,gap,300,-2.20e-01,1.25e-04,1.47e-02,4.20e-03
8,pngOptim,all,vc,improvement,300,-1.00e-01,8.30e-02,1.00e+00,3.20e-01
9,pngOptim,all,vc,gap,300,-2.13e-01,2.07e-04,2.40e-02,4.96e-03


## Correlations that remain significant after Holm's correction

In [3]:
display(corr[corr.p_holm < 0.05])

,software,group,x,y,n,rho,p,p_holm,p_bh
7,pngOptim,all,vt,gap,300,-0.22,1.25e-04,1.47e-02,4.20e-03
9,pngOptim,all,vc,gap,300,-0.21,2.07e-04,2.40e-02,4.96e-03
17,sat4j,all,checkpoint,gap,300,0.26,4.54e-06,5.45e-04,5.45e-04
73,sat4j,FI,vt,gap,75,0.43,1.40e-04,1.64e-02,4.20e-03
80,sat4j,RS,vc,improvement,75,-0.44,6.89e-05,8.20e-03,4.13e-03
83,sat4j,RS,checkpoint,gap,75,0.41,3.08e-04,3.54e-02,6.15e-03
